In [ ]:
import re
import numpy as np
import pandas as pd

_KEYWORD_REPEAT_LIST = ["UNION", "OR", "AND", "CONCAT", "SLEEP", "WAITFOR", "INFORMATION_SCHEMA", "BENCHMARK"]
_KEYWORD_REPEAT_PATTERNS = [re.compile(r"\b" + kw + r"\b") for kw in _KEYWORD_REPEAT_LIST]

_UNION_PATTERN = re.compile(r"\bUNION\b")
_OR_PATTERN = re.compile(r"\bOR\b")
_DROP_PATTERN = re.compile(r"\bDROP\b")
_SELECT_FROM_PATTERN = re.compile(r"\bSELECT\b.*\bFROM\b", re.DOTALL)

FEATURE_NAMES = [
    "length", "uppercase_ratio", "n_whitespace", "n_digit",
    "n_quote", "n_equal", "n_comma", "n_paren", "n_percent", "n_comment_markers",
    "keyword_repeat_count", "n_semicolon",
    "has_union", "has_or", "has_select_from", "has_drop"
]

def extract_features(query: str) -> np.ndarray:
    q = str(query)
    q_upper = q.upper()
    length = len(q)

    # --- 12 cột liên tục (index 0-11) ---
    n_upper_chars = sum(1 for c in q if c.isupper())    
    uppercase_ratio = n_upper_chars / length if length > 0 else 0.0
    n_whitespace = sum(1 for c in q if c.isspace())
    n_digit = sum(1 for c in q if c.isdigit())
    n_quote = q.count("'")
    n_equal = q.count("=")
    n_comma = q.count(",")
    n_paren = q.count("(") + q.count(")")
    n_percent = q.count("%")
    n_comment_markers = q.count("--") + q.count("#") + q.count("/*")
    keyword_repeat_count = sum(len(p.findall(q_upper)) for p in _KEYWORD_REPEAT_PATTERNS)
    n_semicolon = q.count(";")

    # --- 4 cột nhị phân (index 12-15) ---
    has_union = 1 if _UNION_PATTERN.search(q_upper) else 0
    has_or = 1 if _OR_PATTERN.search(q_upper) else 0
    has_select_from = 1 if _SELECT_FROM_PATTERN.search(q_upper) else 0
    has_drop = 1 if _DROP_PATTERN.search(q_upper) else 0

    return np.array([
        length, uppercase_ratio, n_whitespace, n_digit,
        n_quote, n_equal, n_comma, n_paren, n_percent, n_comment_markers,
        keyword_repeat_count, n_semicolon,
        has_union, has_or, has_select_from, has_drop
    ], dtype=float)

In [ ]:
test_queries = [
    "1 UNION SELECT username, password FROM users--",
    "admin' OR '1'='1",
    "SELECT * FROM products WHERE id = 5",
    "'; DROP TABLE users; --",
    "SELECT name FROM employees WHERE department = 'Sales'",
]

for q in test_queries:
    vec = extract_features(q)
    print(q)
    print(dict(zip(FEATURE_NAMES, vec)))
    print()

1 UNION SELECT username, password FROM users--
{'length': np.float64(46.0), 'uppercase_ratio': np.float64(0.32608695652173914), 'n_whitespace': np.float64(6.0), 'n_digit': np.float64(1.0), 'n_quote': np.float64(0.0), 'n_equal': np.float64(0.0), 'n_comma': np.float64(1.0), 'n_paren': np.float64(0.0), 'n_percent': np.float64(0.0), 'n_comment_markers': np.float64(1.0), 'keyword_repeat_count': np.float64(1.0), 'n_semicolon': np.float64(0.0), 'has_union': np.float64(1.0), 'has_or': np.float64(0.0), 'has_select_from': np.float64(1.0), 'has_drop': np.float64(0.0)}

admin' OR '1'='1
{'length': np.float64(16.0), 'uppercase_ratio': np.float64(0.125), 'n_whitespace': np.float64(2.0), 'n_digit': np.float64(2.0), 'n_quote': np.float64(4.0), 'n_equal': np.float64(1.0), 'n_comma': np.float64(0.0), 'n_paren': np.float64(0.0), 'n_percent': np.float64(0.0), 'n_comment_markers': np.float64(0.0), 'keyword_repeat_count': np.float64(1.0), 'n_semicolon': np.float64(0.0), 'has_union': np.float64(0.0), 'has_or

In [ ]:
test_df = pd.read_csv("../datasets/processed/test.csv") 

train_df = pd.read_csv("../datasets/processed/train.csv")
test_df = pd.read_csv("../datasets/processed/test.csv")

train_features = np.array([extract_features(q) for q in train_df["Query"]])
test_features = np.array([extract_features(q) for q in test_df["Query"]])

feat_df = pd.DataFrame(train_features, columns=FEATURE_NAMES)
feat_df["Label"] = train_df["Label"].values

print(feat_df.groupby("Label")[["length", "uppercase_ratio", "n_whitespace", "n_digit",
                                  "n_quote", "n_equal", "n_comma", "n_paren", "n_percent",
                                  "n_comment_markers"]].mean())

np.save("../datasets/processed/train_features_raw.npy", train_features)
np.save("../datasets/processed/test_features_raw.npy", test_features)
print("Đã lưu ma trận đặc trưng thô.")

           length  uppercase_ratio  n_whitespace    n_digit   n_quote  \
Label                                                                   
0       39.894508         0.237200      6.540968   1.707464  0.661503   
1      118.736981         0.003855     43.993078  18.320589  1.128433   

        n_equal   n_comma   n_paren  n_percent  n_comment_markers  
Label                                                              
0      0.208872  0.335424  0.577839   0.030022           0.000640  
1      1.205669  1.808723  7.528895   0.196770           0.525599  
Đã lưu ma trận đặc trưng thô.


In [ ]:
import numpy as np
import json
import joblib
import os
from sklearn.preprocessing import StandardScaler


train_features = np.load("../datasets/processed/train_features_raw.npy")
test_features = np.load("../datasets/processed/test_features_raw.npy")

FEATURE_NAMES = [
    "length", "uppercase_ratio", "n_whitespace", "n_digit",
    "n_quote", "n_equal", "n_comma", "n_paren", "n_percent", "n_comment_markers",
    "keyword_repeat_count", "n_semicolon",
    "has_union", "has_or", "has_select_from", "has_drop"
]

CONTINUOUS_IDX = list(range(0, 12))   
BINARY_IDX = list(range(12, 16))       

#B1 tính ngưỡng percentile 99 trên train
clip_thresholds = {}
for i in CONTINUOUS_IDX:
    col_name = FEATURE_NAMES[i]
    threshold = np.percentile(train_features[:, i], 99)
    clip_thresholds[col_name] = float(threshold)

print("Ngưỡng clip percentile 99 (tính trên train):")
for k, v in clip_thresholds.items():
    print(f"  {k}: {v:.4f}")

#B2 áp dụng clip trên train và test
train_clipped = train_features.copy()
test_clipped = test_features.copy()

for i in CONTINUOUS_IDX:
    col_name = FEATURE_NAMES[i]
    thresh = clip_thresholds[col_name]
    train_clipped[:, i] = np.clip(train_clipped[:, i], a_min=None, a_max=thresh)
    test_clipped[:, i] = np.clip(test_clipped[:, i], a_min=None, a_max=thresh)

# Kiểm tra nhanh max của length trước/sau clip
print("\nlength — max trước clip (train):", train_features[:, 0].max())
print("length — max sau clip (train):", train_clipped[:, 0].max())

#B3 fit StandertScaler trên train đã clip 
scaler = StandardScaler()
scaler.fit(train_clipped[:, CONTINUOUS_IDX])

#B4 transform train và test
train_scaled_continuous = scaler.transform(train_clipped[:, CONTINUOUS_IDX])
test_scaled_continuous = scaler.transform(test_clipped[:, CONTINUOUS_IDX])

#B5 ghép lại với 4 cột nhị phân
train_final = np.hstack([train_scaled_continuous, train_clipped[:, BINARY_IDX]])
test_final = np.hstack([test_scaled_continuous, test_clipped[:, BINARY_IDX]])

print("\ntrain_final shape:", train_final.shape)
print("test_final shape:", test_final.shape)

#B6 lưu artifact
os.makedirs("artifacts", exist_ok=True)

with open("../artifacts/clip_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(clip_thresholds, f, ensure_ascii=False, indent=2)

joblib.dump(scaler, "../artifacts/tier1_scaler.pkl")

with open("artifacts/feature_order.json", "w", encoding="utf-8") as f:
    json.dump(FEATURE_NAMES, f, ensure_ascii=False, indent=2)

#B7 lưu ma trận đặc trưng đã xử lý hoàn chỉnh
np.save("../datasets/processed/train_features_final.npy", train_final)
np.save("../datasets/processed/test_features_final.npy", test_final)

print("\nĐã lưu artifacts/clip_thresholds.json, tier1_scaler.pkl, feature_order.json")
print("Đã lưu datasets/processed/train_features_final.npy, test_features_final.npy")

Ngưỡng clip percentile 99 (tính trên train):
  length: 377.0000
  uppercase_ratio: 0.6364
  n_whitespace: 171.0000
  n_digit: 55.0000
  n_quote: 6.0000
  n_equal: 3.0000
  n_comma: 8.0000
  n_paren: 36.0000
  n_percent: 2.0000
  n_comment_markers: 1.0000
  keyword_repeat_count: 3.0000
  n_semicolon: 1.0000

length — max trước clip (train): 5370.0
length — max sau clip (train): 377.0

train_final shape: (24724, 16)
test_final shape: (6181, 16)

Đã lưu artifacts/clip_thresholds.json, tier1_scaler.pkl, feature_order.json
Đã lưu datasets/processed/train_features_final.npy, test_features_final.npy


In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

train_df = pd.read_csv("../datasets/processed/train.csv")
train_final = np.load("../datasets/processed/train_features_final.npy")
train_labels = train_df["Label"].values

rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, class_weight="balanced", random_state=42, n_jobs=-1
)

scoring=["precision", "recall", "f1"]

start = time.time()
cv_results = cross_validate(
    rf, train_final, train_labels,
    cv=5, scoring=scoring, return_train_score=False
)
elapsed = time.time() - start

print(f"Thời gian chạy cross-validation: {elapsed:.1f} giây")
print()
for metric in scoring:
    scores = cv_results[f"test_{metric}"]
    print(f"{metric}: mean={scores.mean():.4f}, std={scores.std():.4f}")
    print(f"  từng fold: {np.round(scores, 4)}")


start = time.time()
rf.fit(train_final, train_labels)
print(f"\nThời gian fit cuối trên toàn bộ train: {time.time()-start:.1f} giây")

Thời gian chạy cross-validation: 8.2 giây

precision: mean=0.9986, std=0.0007
  từng fold: [0.9978 0.9994 0.9989 0.9978 0.9989]
recall: mean=0.9959, std=0.0010
  từng fold: [0.9962 0.9967 0.9967 0.9962 0.994 ]
f1: mean=0.9972, std=0.0006
  từng fold: [0.997  0.9981 0.9978 0.997  0.9964]

Thời gian fit cuối trên toàn bộ train: 1.5 giây


In [ ]:
# kiểm tra tùng lặp tránh học mẫu có sẵn
query_counts = train_df["Query"].value_counts()
print("Số câu truy vấn xuất hiện >= 2 lần:", (query_counts >= 2).sum())
print("Số câu truy vấn xuất hiện >= 5 lần:", (query_counts >= 5).sum())
print("Top 10 câu xuất hiện nhiều nhất:")
print(query_counts.head(10))

Số câu truy vấn xuất hiện >= 2 lần: 0
Số câu truy vấn xuất hiện >= 5 lần: 0
Top 10 câu xuất hiện nhiều nhất:
Query
1"   )    )     )   or   (  select 2*  (  if   (    (   select * from   (  select concat  (  0x7171706a71,  (  select   (  elt  (  8113  =  8113,1   )    )     )  ,0x717a767a71,0x78   )    )   s  )  , 8446744073709551610, 8446744073709551610   )    )     )   and    (    (     (  "twpn" like "twpn    1
 select * from users where id  =  1 or "1#" or 1  =  1 -- 1                                                                                                                                                                                                                                                  1
1'  )   as sjij where 8659  =  8659                                                                                                                                                                                                                                                    

In [ ]:
import time
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

scoring = ["precision", "recall", "f1"]

start = time.time()
cv_results_svm = cross_validate(
    svm, train_final, train_labels,
    cv=5, scoring=scoring, return_train_score=False
)
elapsed = time.time() - start

print(f"Thời gian chạy cross-validation SVM: {elapsed:.1f} giây")
print()
for metric in scoring:
    scores = cv_results_svm[f"test_{metric}"]
    print(f"{metric}: mean={scores.mean():.4f}, std={scores.std():.4f}")
    print(f"  từng fold: {np.round(scores, 4)}")

c:\Users\Admin\Documents\SQLi-Detection\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Admin\Documents\SQLi-Detection\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Admin\Documents\SQLi-Detection\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Admin\Documents\SQLi-Detection\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probab

Thời gian chạy cross-validation SVM: 39.8 giây

precision: mean=0.9989, std=0.0012
  từng fold: [1.     1.     0.9967 0.9989 0.9989]
recall: mean=0.9912, std=0.0019
  từng fold: [0.9929 0.9918 0.9885 0.9934 0.9896]
f1: mean=0.9950, std=0.0015
  từng fold: [0.9964 0.9959 0.9926 0.9961 0.9942]


In [ ]:
start = time.time()
svm.fit(train_final, train_labels)
print(f"Thời gian fit cuối trên toàn bộ train: {time.time()-start:.1f} giây")

import joblib
import os
os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/tier1_rf.pkl")
joblib.dump(svm, "models/tier1_svm.pkl")
print("Đã lưu tạm models/tier1_rf.pkl và models/tier1_svm.pkl")

c:\Users\Admin\Documents\SQLi-Detection\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Thời gian fit cuối trên toàn bộ train: 12.3 giây
Đã lưu tạm models/tier1_rf.pkl và models/tier1_svm.pkl


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

test_final = np.load("../datasets/processed/test_features_final.npy")
test_df = pd.read_csv("../datasets/processed/test.csv") 
test_labels = test_df["Label"].values

FEATURE_NAMES = [
    "length", "uppercase_ratio", "n_whitespace", "n_digit",
    "n_quote", "n_equal", "n_comma", "n_paren", "n_percent", "n_comment_markers",
    "keyword_repeat_count", "n_semicolon",
    "has_union", "has_or", "has_select_from", "has_drop"
]

#B1 đánh giá trên test.csv
for name, model in [("Random Forest", rf), ("SVM", svm)]:
    y_pred = model.predict(test_final)
    precision = precision_score(test_labels, y_pred)
    recall = recall_score(test_labels, y_pred)
    f1 = f1_score(test_labels, y_pred)
    cm = confusion_matrix(test_labels, y_pred)

    print(f"===== {name} — đánh giá trên test.csv =====")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print(f"  (TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]})")
    print()

#B2 feature importance từ RF
importances = rf.feature_importances_
importance_ranking = sorted(zip(FEATURE_NAMES, importances), key=lambda x: -x[1])

print("===== Feature importance (Random Forest) =====")
for name, score in importance_ranking:
    print(f"{name}: {score:.4f}")

#B3 ma trận tương quan 16 đặc trưng
import pandas as pd

feat_df = pd.DataFrame(train_final, columns=FEATURE_NAMES)
corr_matrix = feat_df.corr()

print("\n===== Các cặp đặc trưng có tương quan > 0.7 (trị tuyệt đối) =====")
seen = set()
for i in range(len(FEATURE_NAMES)):
    for j in range(i+1, len(FEATURE_NAMES)):
        c = corr_matrix.iloc[i, j]
        if abs(c) > 0.7:
            print(f"{FEATURE_NAMES[i]} <-> {FEATURE_NAMES[j]}: {c:.3f}")

===== Random Forest — đánh giá trên test.csv =====
Precision: 0.9978
Recall: 0.9938
F1-score: 0.9958
Confusion Matrix:
[[3900    5]
 [  14 2262]]
  (TN=3900, FP=5, FN=14, TP=2262)

===== SVM — đánh giá trên test.csv =====
Precision: 0.9973
Recall: 0.9903
F1-score: 0.9938
Confusion Matrix:
[[3899    6]
 [  22 2254]]
  (TN=3899, FP=6, FN=22, TP=2254)

===== Feature importance (Random Forest) =====
n_digit: 0.2374
keyword_repeat_count: 0.1703
uppercase_ratio: 0.1329
n_equal: 0.1016
n_whitespace: 0.1005
n_comment_markers: 0.0830
n_paren: 0.0734
length: 0.0370
has_select_from: 0.0158
n_comma: 0.0142
n_quote: 0.0141
has_union: 0.0097
has_or: 0.0060
n_semicolon: 0.0021
n_percent: 0.0018
has_drop: 0.0001

===== Các cặp đặc trưng có tương quan > 0.7 (trị tuyệt đối) =====
length <-> n_whitespace: 0.957
length <-> n_digit: 0.766
length <-> n_paren: 0.906
n_whitespace <-> n_digit: 0.795
n_whitespace <-> n_paren: 0.979
n_digit <-> n_paren: 0.777


In [ ]:
print("===== Feature importance đầy đủ (Random Forest) =====")
for name, score in importance_ranking:
    print(f"{name}: {score:.4f}")

===== Feature importance đầy đủ (Random Forest) =====
n_digit: 0.2374
keyword_repeat_count: 0.1703
uppercase_ratio: 0.1329
n_equal: 0.1016
n_whitespace: 0.1005
n_comment_markers: 0.0830
n_paren: 0.0734
length: 0.0370
has_select_from: 0.0158
n_comma: 0.0142
n_quote: 0.0141
has_union: 0.0097
has_or: 0.0060
n_semicolon: 0.0021
n_percent: 0.0018
has_drop: 0.0001


In [ ]:
print("===== Tương quan keyword_repeat_count với các cột từ khóa binary =====")
for col in ["has_union", "has_or", "has_select_from", "has_drop"]:
    c = corr_matrix.loc["keyword_repeat_count", col]
    print(f"keyword_repeat_count <-> {col}: {c:.3f}")

print("\n===== Toàn bộ cặp tương quan > 0.5 (để không bỏ sót) =====")
for i in range(len(FEATURE_NAMES)):
    for j in range(i+1, len(FEATURE_NAMES)):
        c = corr_matrix.iloc[i, j]
        if abs(c) > 0.5:
            print(f"{FEATURE_NAMES[i]} <-> {FEATURE_NAMES[j]}: {c:.3f}")

===== Tương quan keyword_repeat_count với các cột từ khóa binary =====
keyword_repeat_count <-> has_union: 0.234
keyword_repeat_count <-> has_or: 0.521
keyword_repeat_count <-> has_select_from: 0.046
keyword_repeat_count <-> has_drop: -0.024

===== Toàn bộ cặp tương quan > 0.5 (để không bỏ sót) =====
length <-> n_whitespace: 0.957
length <-> n_digit: 0.766
length <-> n_equal: 0.673
length <-> n_paren: 0.906
length <-> keyword_repeat_count: 0.594
uppercase_ratio <-> has_select_from: 0.642
n_whitespace <-> n_digit: 0.795
n_whitespace <-> n_equal: 0.668
n_whitespace <-> n_paren: 0.979
n_whitespace <-> keyword_repeat_count: 0.613
n_digit <-> n_equal: 0.572
n_digit <-> n_paren: 0.777
n_digit <-> keyword_repeat_count: 0.591
n_equal <-> n_paren: 0.570
n_equal <-> keyword_repeat_count: 0.624
n_paren <-> keyword_repeat_count: 0.546
keyword_repeat_count <-> has_or: 0.521
